<a href="https://colab.research.google.com/github/tk-learning-artifacts/ai-accelerator-2026-notebooks/blob/main/student/x-diffusion-2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Beyond the course · Diffusion 2 — A denoiser that sees space

In [ ]:
# Installs what Colab doesn't ship with.
!pip install -q medmnist

In Diffusion 1 your denoiser was Wednesday's kind of network: it saw each
8×8 digit as a list of 64 numbers and had no idea which pixel was next to
which. Real image generators use a convolutional network, usually a
**U-Net**, the same shape you trained for nuclei in Segmentation 1. In this
notebook you give a U-Net the step number, train it to remove noise from
bigger pictures, and generate handwritten digits or, if you choose, tissue
patches.

This is an extension pathway: you work through it alone or with a partner,
at your own pace. You need Diffusion 1, and Thursday's CNN notebook (or
Segmentation 1, for the U-Net). Allow about two hours.

How it works, as in class: 🔮 predict before you run, ✏️ replace the `...`,
✅ run the check under it. A check that fails tells you what to look at. Each
section ends with a **You're done when** line.

**GPU or not?** Section 3 trains the model. With a GPU (Runtime → Change
runtime type → T4 GPU) it trains on full 28×28 pictures for longer. Without
one it shrinks the pictures to 16×16 and trains a shorter run: for digits,
about five minutes on the free CPU, with blurrier but recognisable results.
The tissue option in section 5 is slower: plan for 10 to 15 minutes of
training on the free CPU, plus a 206 MB download (usually under a minute).
Nothing downloads until you run a ⬇️ download cell in section 1, and only the
one for your choice does anything.

In [ ]:
import os
import hashlib
import pathlib
import shutil
import urllib.request
import time
import numpy as np
import matplotlib.pyplot as plt
import torch
from torch import nn
import torch.nn.functional as F
import torchvision


def is_done(*values):
    """True once none of the values is still `...`."""
    return all(v is not ... for v in values)


def check(ok, hint):
    if not ok:
        raise AssertionError("Not yet. " + hint)
    print("✅ Correct!")


def fetch_from_mirror(url, dest, sha256):
    """Download url to dest and check its sha256. Returns True if the file is good."""
    # The original data servers get slow when a whole class downloads at once,
    # so we keep a copy of each file on our own fast server.
    dest = pathlib.Path(dest)
    dest.parent.mkdir(parents=True, exist_ok=True)

    def good(path):
        with open(path, "rb") as f:
            return hashlib.file_digest(f, "sha256").hexdigest() == sha256

    if dest.exists() and good(dest):
        return True
    # Download to a .part file first, so a stopped download never looks finished.
    part = dest.with_name(dest.name + ".part")
    try:
        # Our server turns away requests that look like they come from a robot,
        # so we give ours a name.
        request = urllib.request.Request(url, headers={"User-Agent": "tk-accelerator-notebooks"})
        with urllib.request.urlopen(request, timeout=30) as reply, open(part, "wb") as out:
            shutil.copyfileobj(reply, out)
        if not good(part):
            raise ValueError("the checksum does not match")
        part.replace(dest)
        return True
    except Exception:
        part.unlink(missing_ok=True)
        print("Our copy of the data did not download, so we are using the original source.")
        return False


class DataMissing(Exception):
    """Stops a cell with one plain message instead of a long error."""

    def __init__(self, message):
        super().__init__(message)
        self.message = message

    def _render_traceback_(self):
        return [self.message]


def require_data(name, *paths):
    """Stop with a plain message unless the downloaded files are on disk."""
    if not all(pathlib.Path(p).exists() for p in paths):
        raise DataMissing(f"The {name} data isn't downloaded yet. Run the ⬇️ Download cell above first.")


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
GPU = device.type == "cuda"
# The instructors' checker sets NB_CHECK to run this notebook quickly; ignore it.
QUICK = os.environ.get("NB_CHECK") == "1"
torch.manual_seed(0)
np.random.seed(0)
print("Running on", device)

## 1. Bigger pictures, same noise

Choose what your model will learn to draw. `"digits"` is MNIST: 60,000
handwritten digits, 28×28, black and white. `"tissue"` is PathMNIST from
Friday: 90,000 colour patches of nine tissue types (a bigger download, 206
MB, usually under a minute). Start with digits; you can come back and switch.

In [ ]:
DATASET = "digits"  # @param ["digits", "tissue"]
SIZE = 28 if GPU else 16   # without a GPU, pictures are shrunk to 16×16
print("You chose:", DATASET)

Each choice has its own download cell, and the one you did not choose does
nothing. If you change `DATASET` later, run its download cell too.

⬇️ **Download the digits (12 MB).** Only run the next cell if you chose `"digits"`. It downloads 12 MB, so skip it if you are only reading. It is safe to run twice. "Run all" runs it too, but it does nothing when you chose `"tissue"`.

In [ ]:
# ⬇️ Download cell: MNIST digits (12 MB)
if "DATASET" not in globals():
    raise DataMissing("Run the cell that sets DATASET first, then this one.")
elif DATASET != "digits":
    print('You chose "tissue", so the digits are not downloaded.')
elif pathlib.Path("data/MNIST/raw/train-images-idx3-ubyte").exists():
    print("MNIST is already downloaded. Nothing to do.")
else:
    torchvision.datasets.MNIST("data", train=True, download=True)
    print("Done: MNIST downloaded (12 MB).")

⬇️ **Download the tissue (206 MB).** Only run the next cell if you chose `"tissue"`. It downloads 206 MB, so skip it if you are only reading. It is safe to run twice. "Run all" runs it too, but it does nothing when you chose `"digits"`.

In [ ]:
# ⬇️ Download cell: PathMNIST tissue (206 MB)
import medmnist

if "DATASET" not in globals():
    raise DataMissing("Run the cell that sets DATASET first, then this one.")
elif DATASET != "tissue":
    print('You chose "digits", so the tissue is not downloaded. To use it, set DATASET = "tissue" above.')
elif pathlib.Path("data/pathmnist.npz").exists():
    print("PathMNIST is already downloaded. Nothing to do.")
else:
    if not fetch_from_mirror("https://data.tk.sg/pathmnist/pathmnist.npz", "data/pathmnist.npz",
                             "81823f52dc622e69db2db4c72f8e8e617938dd6864d3c1f23d4e49724a28ea72"):
        medmnist.PathMNIST(split="train", download=True, root="data")
    print("Done: PathMNIST downloaded (206 MB).")

In [ ]:
if DATASET == "digits":
    require_data("MNIST", "data/MNIST/raw/train-images-idx3-ubyte")
    data = torchvision.datasets.MNIST("data", train=True, download=False)
    pictures = torch.tensor(data.data.numpy(), dtype=torch.float32)[:, None] / 255
else:
    require_data("PathMNIST", "data/pathmnist.npz")
    data = medmnist.PathMNIST(split="train", download=False, root="data")
    pictures = torch.tensor(data.imgs, dtype=torch.float32).permute(0, 3, 1, 2) / 255
if SIZE != 28:
    pictures = F.interpolate(pictures, size=SIZE, mode="bilinear", antialias=True)
pictures = pictures * 2 - 1          # from 0–1 to −1–1, as in Diffusion 1
CHANNELS = pictures.shape[1]
print(DATASET, tuple(pictures.shape))


def show(batch, rows=1, size=1.2):
    """Draw a batch of pictures (values −1 to 1) in a grid."""
    batch = batch.detach().cpu().clamp(-1, 1)
    grid = torchvision.utils.make_grid(batch, nrow=int(np.ceil(len(batch) / rows)), normalize=True, value_range=(-1, 1), pad_value=1)
    plt.figure(figsize=(size * len(batch) / rows, size * rows))
    plt.imshow(grid.permute(1, 2, 0))
    plt.axis("off")
    plt.show()


show(pictures[:12])

The noise schedule is Diffusion 1's: 200 steps, and `keep[t]` says how much
of the picture is left at step t.

In [ ]:
T = 200
betas = torch.linspace(1e-4, 0.05, T).to(device)
keep = torch.cumprod(1 - betas, dim=0)

In Diffusion 1 a batch of pictures had shape (number of pictures, 64), so
`keep[t][:, None]` made one number per picture into a column that multiplied
each picture's 64 pixels. Now a batch has shape (number, channels, height,
width): four dimensions.

✏️ **Exercise 1a.** Write `add_noise(clean, t, noise)` for four-dimensional
batches. Reshape `keep[t]` so it has shape (number, 1, 1, 1), then use the same
formula as Diffusion 1: √keep × clean + √(1 − keep) × noise.

In [ ]:
def add_noise(clean, t, noise):
    k = ...
    return ...

In [ ]:
c = torch.ones(3, CHANNELS, SIZE, SIZE, device=device)
t3 = torch.tensor([0, 100, 199], device=device)
try:
    out = add_noise(c, t3, torch.zeros_like(c))
except RuntimeError as err:
    raise AssertionError("Not yet. The shapes don't fit together (" + str(err)[:60] + "...). "
                         "keep[t] has one number per picture: reshape it to (number, 1, 1, 1) with [:, None, None, None].")
check(is_done(out), "add_noise still returns `...`.")
check(out.shape == c.shape, f"The result should keep the batch's shape {tuple(c.shape)}, not {tuple(out.shape)}. Reshape keep[t] to (number, 1, 1, 1).")
check(torch.allclose(out[:, 0, 0, 0], keep[t3].sqrt()), "With zero noise, an all-ones picture should become √keep[t] everywhere.")
check(torch.allclose(add_noise(torch.zeros_like(c), t3, c)[:, 0, 0, 0], (1 - keep[t3]).sqrt()), "With a blank picture, the result should be √(1 − keep[t]) × noise.")

In [ ]:
t_demo = torch.tensor([0, 25, 50, 100, 150, 199], device=device)
first = pictures[:1].to(device).repeat(6, 1, 1, 1)
show(add_noise(first, t_demo, torch.randn_like(first)))

**You're done when** your four-dimensional `add_noise` passes its check and
shows one picture dissolving.

## 2. A U-Net that knows the step

The denoiser is a small U-Net, like Segmentation 1's, with one new idea. In
Diffusion 1 the step number went in as 32 extra inputs next to the pixels.
A convolution has no "next to the pixels". Instead, every block turns the 32
step features into one number per channel, and **adds that number to every
pixel of its channel**. The block's feature maps are shifted up or down
depending on how noisy the picture is.

In [ ]:
def step_features(t):
    """32 numbers describing each step number in t (as in Diffusion 1)."""
    speeds = torch.exp(-torch.arange(16, device=t.device) * 0.5) * 50
    angle = t[:, None].float() / T * speeds[None, :]
    return torch.cat([angle.sin(), angle.cos()], dim=1)


per_channel = nn.Linear(32, 8).to(device)(step_features(torch.tensor([5, 150], device=device)))
print("one number per channel, for 2 pictures:", tuple(per_channel.shape))

🔮 **Predict.** A feature map has shape (2, 8, 16, 16): 2 pictures, 8 channels,
16×16. The step gives (2, 8). To add one number to every pixel of a channel,
what shape must the step's numbers be turned into first?

In [ ]:
my_prediction = ""  # @param {type:"string"}

✏️ **Exercise 2a.** Finish `StepBlock.forward`: after the first convolution
(`h`), add `self.time(step_feats)` reshaped to (number, channels, 1, 1).

In [ ]:
class StepBlock(nn.Module):
    """Two 3×3 convolutions, with the step added after the first."""

    def __init__(self, c_in, c_out):
        super().__init__()
        self.conv1 = nn.Conv2d(c_in, c_out, 3, padding=1)
        self.conv2 = nn.Conv2d(c_out, c_out, 3, padding=1)
        self.norm1, self.norm2 = nn.GroupNorm(8, c_out), nn.GroupNorm(8, c_out)
        self.time = nn.Linear(32, c_out)

    def forward(self, x, step_feats):
        h = F.silu(self.norm1(self.conv1(x)))
        h = ...
        return F.silu(self.norm2(self.conv2(h)))

`GroupNorm` rescales each group of channels to an average of 0 and a spread
of 1. Batch norm (from the competition notebook) does the same across a whole
batch; GroupNorm does it inside each picture rather than across the
batch, which suits diffusion, where every picture in a batch is at a
different step.

In [ ]:
blk = StepBlock(CHANNELS, 8).to(device)
feats = step_features(torch.tensor([5, 150], device=device))
try:
    out = blk(torch.zeros(2, CHANNELS, 16, 16, device=device), feats)
except TypeError:
    out = ...
except RuntimeError:
    raise AssertionError("Not yet. The step's numbers have shape (2, 8) and h has shape (2, 8, 16, 16), so they "
                         "can't be added as they are. Reshape with [:, :, None, None] first.")
check(is_done(out), "forward still has a `...`. Add the step's numbers to h.")
check(tuple(out.shape) == (2, 8, 16, 16), f"The block should give (2, 8, 16, 16), not {tuple(out.shape)}.")
check(not torch.allclose(out[0], out[1]), "Two blank pictures at different steps should come out different: is the step being added?")

Everyone continues from here (code hidden; double-click to see it after). It
also defines the U-Net, which is Segmentation 1's with `StepBlock`s and two
levels.

In [ ]:
#@title Run this to continue { display-mode: "form" }
def add_noise(clean, t, noise):
    share = keep[t].view(-1, 1, 1, 1)
    return share.sqrt() * clean + (1 - share).sqrt() * noise


class StepBlock(nn.Module):
    def __init__(self, c_in, c_out):
        super().__init__()
        self.conv1 = nn.Conv2d(c_in, c_out, 3, padding=1)
        self.conv2 = nn.Conv2d(c_out, c_out, 3, padding=1)
        self.norm1, self.norm2 = nn.GroupNorm(8, c_out), nn.GroupNorm(8, c_out)
        self.time = nn.Linear(32, c_out)

    def forward(self, x, step_feats):
        h = F.silu(self.norm1(self.conv1(x))) + self.time(step_feats).view(len(x), -1, 1, 1)
        return F.silu(self.norm2(self.conv2(h)))


class DiffusionUNet(nn.Module):
    def __init__(self, c=32):
        super().__init__()
        self.down1, self.down2, self.down3 = StepBlock(CHANNELS, c), StepBlock(c, 2 * c), StepBlock(2 * c, 2 * c)
        self.up2, self.join2 = nn.ConvTranspose2d(2 * c, 2 * c, 2, stride=2), StepBlock(4 * c, 2 * c)
        self.up1, self.join1 = nn.ConvTranspose2d(2 * c, c, 2, stride=2), StepBlock(2 * c, c)
        self.head = nn.Conv2d(c, CHANNELS, 1)

    def forward(self, x, t):
        f = step_features(t)
        x1 = self.down1(x, f)
        x2 = self.down2(F.avg_pool2d(x1, 2), f)
        x3 = self.down3(F.avg_pool2d(x2, 2), f)
        y = self.join2(torch.cat([self.up2(x3), x2], dim=1), f)
        y = self.join1(torch.cat([self.up1(y), x1], dim=1), f)
        return self.head(y)


net = DiffusionUNet()
print(f"numbers to learn: {sum(p.numel() for p in net.parameters()):,}")

**You're done when** your `StepBlock` passes its check and you can say how
the step number reaches a convolution.

## 3. Train it

The training step is Diffusion 1's, with pictures instead of lists: pick
random pictures, random steps and fresh noise; mix them with `add_noise`; the
loss is the mean squared error between the U-Net's guess and the real noise.

🔮 **Predict.** In Diffusion 1 the loss levelled off around 0.2. Will this
model's loss level off higher or lower? (Hint: a convolution sees each
pixel's neighbours. How does that help to tell noise from pen strokes?)

In [ ]:
my_prediction = ""  # @param {type:"string"}

In [ ]:
STEPS = 20 if QUICK else (6000 if GPU else 1500)
BATCH = 128 if GPU else 64
model = DiffusionUNet().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=2e-3)
losses, start = [], time.time()
for step in range(STEPS):
    clean = pictures[torch.randint(len(pictures), (BATCH,))].to(device)
    t = torch.randint(T, (BATCH,), device=device)
    noise = torch.randn_like(clean)
    optimizer.zero_grad()
    loss = F.mse_loss(model(add_noise(clean, t, noise), t), noise)
    loss.backward()
    optimizer.step()
    losses.append(loss.item())
    if step % 250 == 0 or step == STEPS - 1:
        print(f"step {step:5d}  loss {loss.item():.3f}  {time.time() - start:.0f} s")

plt.figure(figsize=(6, 3))
plt.plot(np.convolve(losses, np.ones(20) / 20, mode="valid"))
plt.xlabel("step"); plt.ylabel("loss (average of 20)"); plt.yscale("log")
plt.show()

**You're done when** the loss has levelled off (around 0.05 to 0.06 for
digits, a little lower for tissue).

## 4. Generate, and check for copying

The reverse step is Diffusion 1's, working on whole pictures.

In [ ]:
@torch.no_grad()
def generate(model, n):
    model.eval()
    x = torch.randn(n, CHANNELS, SIZE, SIZE, device=device)
    for t in reversed(range(T)):
        guess = model(x, torch.full((n,), t, device=device))
        x = (x - betas[t] / (1 - keep[t]).sqrt() * guess) / (1 - betas[t]).sqrt()
        if t > 0:
            x = x + betas[t].sqrt() * torch.randn_like(x)
    return x.cpu()


started = time.time()
samples = generate(model, 32)
print(f"32 pictures in {time.time() - started:.0f} s")
show(samples, rows=4)

Are these new, or has the model memorised some training pictures? Find each
generated picture's **closest training picture**: the one with the smallest
sum of squared pixel differences.

Here is the distance from one generated picture to the first three training
pictures, as a worked example:

In [ ]:
one = samples[0]
for j in range(3):
    print(f"training picture {j}: distance {((pictures[j] - one) ** 2).sum():.1f}")

✏️ **Exercise 4a.** Write `closest(sample, pool)`: the index of the picture in
`pool` (a batch of pictures) with the smallest sum of squared differences
from `sample`. Do it without a loop: `pool - sample` subtracts `sample` from
every picture at once; sum over dimensions 1, 2 and 3; then `argmin()`.

In [ ]:
def closest(sample, pool):
    return ...

In [ ]:
toy = torch.stack([torch.zeros(1, 4, 4), torch.ones(1, 4, 4), torch.full((1, 4, 4), 0.4)])
check(is_done(closest(torch.full((1, 4, 4), 0.9), toy)), "closest still returns `...`.")
check(closest(torch.full((1, 4, 4), 0.9), toy) == 1, "A picture of 0.9s is closest to the picture of 1s (index 1).")
check(closest(torch.full((1, 4, 4), 0.3), toy) == 2, "A picture of 0.3s is closest to the picture of 0.4s (index 2).")
check(isinstance(closest(toy[0], toy), int), "Return a plain whole number: wrap the answer in int().")

In [ ]:
pool = pictures[:5000]
pairs = torch.cat([torch.stack([s, pool[closest(s, pool)]]) for s in samples[:8]])
show(pairs, rows=1)
print("pairs: each generated picture, then its closest of 5,000 training pictures")

🤔 **Think.** Are the pairs copies, or similar-looking but different? Why
does this matter for a generator trained on people's art, or on patients'
scans?

**You're done when** you have generated pictures and an answer to "copies or
not?" based on the pairs.

## 5. Generated tissue: what is it good for?

Set `DATASET = "tissue"` in section 1, then **Runtime → Run all**. The model
learns to draw colour tissue patches instead of digits. (On a CPU, at 16×16,
they look like smudges of the right colours; on a GPU at 28×28 you can tell
some tissue types apart.)

**This takes a while.** Run all downloads 206 MB (the tissue download cell
from section 1) and trains the model again from the start. On the free CPU, plan for 10 to 15 minutes before
anything new appears. Leave the tab open while it runs: read the question
below and write your prediction.

🔮 **Predict.** A Project I team has 500 tumour patches and 5,000 normal ones.
They think about training a diffusion model on the tumour patches and adding
5,000 generated ones. Write down one way this could help and one way it could
go wrong.

In [ ]:
my_prediction = ""  # @param {type:"string"}

**You're done when** you have generated tissue patches (or chosen not to
wait for the download) and written down your help-or-harm answer.

### Stretch: make it yours

- **Ask for a tissue.** Add Diffusion 1's labels and classifier-free guidance
  to this U-Net. PathMNIST has nine tissue types, so each picture gets nine
  more numbers (a one-hot vector), or ten with a "no label" slot, joined onto
  its 32 `step_features`. That makes the vector longer, so every
  `self.time = nn.Linear(32, c_out)` must take the new length (42 with "no
  label") instead of 32. Then generate each of the nine tissue types.
- **Fewer steps when generating.** Generate with every second step only
  (100 of the 200): t = 199, 197, …, 3, 1. For a jump from step t to step
  t − 2, use beta = 1 − keep[t] / keep[t − 2] in the reverse step, and
  `keep[t]` as before. Watch the last jump, from t = 1: there is no step
  −1, and Python will not warn you, because `keep[-1]` quietly reads the
  *last* entry, `keep[199]`. For that jump use 1 in place of keep[t − 2]
  (a clean picture keeps all of itself), and add no fresh noise after it.
  What happens to quality?
- **Bigger.** On a GPU, try `DiffusionUNet(c=64)` and twice the steps.

In [ ]:
# Your stretch work here.